In [ ]:
#Kalyani_036
import os
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.models import Sequential

# ============================================================
# 1. Download Dataset via kagglehub & Load into DataFrame
# ============================================================
print("Downloading dataset...")
dataset_dir = kagglehub.dataset_download("arockiaselciaa/creditcardcsv")
print("Path to dataset files:", dataset_dir)

# Identify the CSV file in the downloaded path
csv_file = os.path.join(dataset_dir, "creditcard.csv")
if not os.path.exists(csv_file):
  csv_file = [
      os.path.join(dataset_dir, f)
      for f in os.listdir(dataset_dir)
      if f.endswith(".csv")
  ][0]

df = pd.read_csv(csv_file)
print("\nDataset Shape:", df.shape)
print("\nClass Distribution:")
print(df["Class"].value_counts())

# ============================================================
# 2. Data Preprocessing & Feature Scaling
# ============================================================
# Standardize the 'Amount' feature and drop the unneeded 'Time' column
scaler = StandardScaler()
df["normAmount"] = scaler.fit_transform(df[["Amount"]])

X = df.drop(columns=["Time", "Amount", "Class"]).values
y = df["Class"].values

# ============================================================
# 3. Stratified Train-Test Split (80% Train, 20% Test)
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# ============================================================
# 4. Compute Class Weights for Extreme Imbalance
# ============================================================
raw_weights = compute_class_weight(
    class_weight="balanced", classes=np.unique(y_train), y=y_train
)
class_weights = {0: raw_weights[0], 1: raw_weights[1]}
print("\nComputed Balanced Class Weights:", class_weights)

# ============================================================
# 5. Build Deep Neural Network Architecture
# ============================================================
model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(16, activation="relu"),
    Dropout(0.2),
    Dense(1, activation="sigmoid"),
])

# ============================================================
# 6. Compile & Train Model
# ============================================================
model.compile(
    optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"]
)

history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=2048,
    validation_split=0.1,
    class_weight=class_weights,
    verbose=1,
)

# ============================================================
# 7. Evaluate Model & Display Metrics
# ============================================================
y_pred_probs = model.predict(X_test, batch_size=2048, verbose=0)
y_pred = (y_pred_probs > 0.5).astype(int).flatten()

print("\n" + "=" * 55)
print("CLASSIFICATION REPORT")
print("=" * 55)
print(classification_report(y_test, y_pred, digits=4, zero_division=0))

# ============================================================
# 8. Confusion Matrix Visualization
# ============================================================
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Non-Fraud (0)", "Fraud (1)"],
    yticklabels=["Non-Fraud (0)", "Fraud (1)"],
)
plt.xlabel("Predicted Label", fontweight="bold")
plt.ylabel("Actual Label", fontweight="bold")
plt.title(
    "Credit Card Fraud Detection - Confusion Matrix",
    fontsize=12,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

Using Colab cache for faster access to the 'creditcardcsv' dataset.
Path to dataset files: /kaggle/input/creditcardcsv

Dataset Shape: (284807, 31)

Class Distribution:
Class
0    284315
1       492
Name: count, dtype: int64

Computed Balanced Class Weights: {0: np.float64(0.5008661206149896), 1: np.float64(289.14340101522845)}
Epoch 1/10
101/101 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.5991 - loss: 0.4294 - val_accuracy: 0.9659 - val_loss: 0.3659
Epoch 2/10
101/101 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.9272 - loss: 0.2848 - val_accuracy: 0.9823 - val_loss: 0.2141
Epoch 3/10
101/101 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.9524 - loss: 0.2327 - val_accuracy: 0.9834 - val_loss: 0.1645
Epoch 4/10
101/101 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9652 - loss: 0.2077 - val_accuracy: 0.9835 - val_loss: 0.1412
Epoch 5/10
